In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "suda2004piagetian")
starting_point = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat
import re


suda_2006 = [['liquid conservation_exp1', '1' ], 
            ['liquid conservation_exp2', '2'],
            ['liquid conservation_exp3', '3'],
            ['liquid conservation_exp4', '4']]

for  file_exp, exp_no  in suda_2006: 
    for dirpath, dirnames, filenames in os.walk(starting_point):
        for filename in [f for f in filenames if f.startswith(file_exp) and  f.endswith("sav")]:
            sav_filepath = os.path.join(dirpath, filename)
            ##get new file names
            csv_filename = filename.replace(".sav", ".csv")
            csv_filepath = os.path.join(dirpath, csv_filename)
            ##write to csv
            read_file = pd.read_spss(sav_filepath, usecols=None, convert_categoricals=True)
            read_file = read_file.assign(experiment=exp_no)
            read_file.to_csv(csv_filepath, encoding='utf-8-sig', index=False)


In [3]:
fulldf=[]
for dirpath, dirnames, filenames in os.walk(starting_point):
    for filename in [f for f in filenames if f.endswith(".csv")]:
        new_csv_filepath = os.path.join(dirpath, filename)
        # print(xlsx_filepath)
        fulldf.append(pd.read_csv(new_csv_filepath))

In [4]:
for index, x in enumerate(fulldf):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    # x=x.rename(columns={"subject": "ape"})
    # x['data_subset']="data_subset_" + str(index+1)
    x['study_id']="suda2004piagetian"
    fulldf[index]=x
fulldf = pd.concat(fulldf, ignore_index=True, sort=False)

In [5]:
fulldf.rename(columns={"subject": "participant", "species":"species_original"}, inplace=True)

In [6]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

In [7]:
# fulldf.columns
output_no_1 = 1 
repeat_1 = 0 
output_no_2 = 1 
repeat_2 = 0 
output_no_3 = 1 
repeat_3 = 0 
output_no_4 = 1 
repeat_4 = 0 
temp = []
for index, row in fulldf.iterrows():
    if row['experiment'] == 1:
        temp.append(output_no_1) 
        repeat_1 = repeat_1+1 
        if repeat_1 == 10: 
            output_no_1 = output_no_1+1 
            repeat_1 = 0
        if output_no_1 == 13:
            output_no_1 = 1 
    elif row['experiment'] == 2:
        temp.append(output_no_2) 
        repeat_2 = repeat_2+1 
        if repeat_2 == 15: 
            output_no_2 = output_no_2+1 
            repeat_2 = 0
        if output_no_2 == 9:
            output_no_2 = 1 
    elif row['experiment'] == 3:
        temp.append(output_no_3) 
        repeat_3 = repeat_3+1 
        if repeat_3 == 10: 
            output_no_3 = output_no_3+1 
            repeat_3 = 0
        if output_no_3 == 7:
            output_no_3 = 1 
    elif row['experiment'] == 4:
        temp.append(output_no_4) 
        repeat_4 = repeat_4+1 
        if repeat_4 == 20: 
            output_no_4 = output_no_4+1 
            repeat_4 = 0
        if output_no_4 == 7:
            output_no_4 = 1 
    else:
        temp.append("") 
fulldf = fulldf.assign(session=temp)

In [8]:
complete_path_age = os.path.join(starting_point, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf = fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age_y": "age_in_years"}, inplace=True) 

space_removal_list = ['am1','am2','con2','side2','fin2','ldlt','fin1','lslm','type']
for x in space_removal_list:
    fulldf[x].replace(' ', '_', inplace=True, regex=True)

In [9]:
fulldf['date'] = fulldf['date'].astype(str)

fulldf['year'] = fulldf['date'].str.slice(0,1)
fulldf['year'] = '200' + fulldf['year'].astype(str)
fulldf['month'] = fulldf['date'].str.slice(1,3)
fulldf['day'] = fulldf['date'].str.slice(3,5)


fulldf['year'].replace('200n', np.nan, inplace=True)
fulldf['month'].replace('an', np.nan, inplace=True)
# fulldf['month'].unique()

In [10]:
fulldf.rename(columns={"type": "trial_type",
                       "lclt":"lc_or_lt_trial",
                       "am1":"1st_amount_chosen",
                       "am2":"2nd_amount_chosen",
                       "con1":"1st_choice_container",
                       "con2":"2nd_choice_container",
                       "side1":"1st_choice_side",
                       "side2":"2nd_choice_side",
                       "fin1":"1st_choice_hesitation",
                       "fin2":"2nd_choice_hesitation",
                       "ldlt":"ld_or_lt_trial",
                       'lslm':'ls_or_lm_trial'}, inplace=True) 


In [11]:
# comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
# ape_dob = pd.read_csv(comp_path_birth_dates) 
# fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
# fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
# fulldf['dodc'].replace('nan-nan-', np.nan, inplace=True)
# fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
# fulldf['dob'] = pd.to_datetime(fulldf['dob'])

# fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

# fulldf['age_original'].unique()

In [12]:



# fulldf['age_original'].unique()

In [13]:
# replace_list = [["age_original",],
#                 ["trial_type",],
#                 ["lc_or_lt_trial",],
#                 ["ld_or_lt_trial"],
#                 ["ls_or_lm_trial",],
#                 ["1st_amount_chosen",],
#                 ["2nd_amount_chosen",],
#                 ["1st_choice_container",],
#                 ["2nd_choice_container",],
#                 ["1st_choice_side",],
#                 ["2nd_choice_side",],
#                 ["1st_choice_hesitation",],
#                 ["2nd_choice_hesitation",],]
# for x,y,k in replace_list:
#     fulldf[x].replace(y, k, inplace=True, regex=True)

In [14]:
fulldf = fulldf[['study_id','experiment' , 'year','month','day', 'participant',
                 'age_in_years','sex', 'species' ,'session', 'trial', 'trial_type',
       'lc_or_lt_trial','ld_or_lt_trial', 'ls_or_lm_trial',
         '1st_amount_chosen', '2nd_amount_chosen',
       '1st_choice_container', '2nd_choice_container', 
       '1st_choice_side','2nd_choice_side', 
         '1st_choice_hesitation', '2nd_choice_hesitation' ]]

In [15]:
fulldf["lc_or_lt_trial"].replace(' ', '_', inplace=True, regex=True)
fulldf["lc_or_lt_trial"].replace('ld_trial', 'lc_trial', inplace=True, regex=True)
fulldf["1st_choice_container"].replace(' ', '_', inplace=True, regex=True)



In [16]:
for index in range(1,5):
    exp = fulldf[fulldf['experiment'] == index]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'suda2004piagetian_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'suda2004piagetian_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)